# Chat Template & 데이터 형식 심화

> Phase 2에서 3가지 형식을 비교했다면, Phase 3에서는 **직접 변환하고 적용**한다

In [1]:
# === 환경 설치 ===
!pip install transformers datasets jinja2

In [2]:
from transformers import AutoTokenizer
from datasets import load_dataset

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. Chat Template 직접 구성해보기

모델이 "누가 말했는지"를 구분하는 형식. 같은 대화를 3가지 형식으로 만들어본다.

In [3]:
# === 같은 대화를 3가지 형식으로 표현 ===
# 파인튜닝 대상 모델에 따라 형식이 달라진다

system_msg = "You are a helpful AI assistant."
user_msg = "What is Python?"
assistant_msg = "Python is a high-level programming language known for its simplicity."

# 1) ChatML (Qwen, Yi 계열)
chatml = f"""<|im_start|>system
{system_msg}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
{assistant_msg}<|im_end|>"""

# 2) Llama 3 형식
llama3 = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_msg}<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_msg}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{assistant_msg}<|eot_id|>"""

# 3) Mistral 형식 (system 메시지 별도 없음)
mistral = f"""[INST] {user_msg} [/INST]{assistant_msg}</s>"""

print("=" * 60)
print("1. ChatML (Qwen, Yi):")
print(chatml)
print("\n" + "=" * 60)
print("2. Llama 3:")
print(llama3)
print("\n" + "=" * 60)
print("3. Mistral:")
print(mistral)
print("\n-> 같은 대화인데 형식이 완전히 다름")
print("   잘못된 형식 = 모델이 역할 구분 실패 = 품질 급락")

1. ChatML (Qwen, Yi):
<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
Python is a high-level programming language known for its simplicity.<|im_end|>

2. Llama 3:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is Python?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Python is a high-level programming language known for its simplicity.<|eot_id|>

3. Mistral:
[INST] What is Python? [/INST]Python is a high-level programming language known for its simplicity.</s>

-> 같은 대화인데 형식이 완전히 다름
   잘못된 형식 = 모델이 역할 구분 실패 = 품질 급락


---
## 2. apply_chat_template() 실전

HuggingFace 토크나이저에 내장된 Jinja2 템플릿으로 자동 변환한다.

In [4]:
# === apply_chat_template 사용법 ===
# messages 리스트를 모델에 맞는 형식으로 자동 변환

# 표준 messages 형식 (OpenAI API와 동일 구조)
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a high-level programming language."},
]

# GPT-2는 chat template이 없으므로 Qwen을 예시로 사용
# (모델 가중치 안 받고 토크나이저만 로드)
try:
    qwen_tok = AutoTokenizer.from_pretrained('Qwen/Qwen2-0.5B-Instruct', trust_remote_code=True)
    result = qwen_tok.apply_chat_template(messages, tokenize=False)
    print("Qwen2 Chat Template 결과:")
    print(result)
except Exception as e:
    # 네트워크 문제 시 수동으로 ChatML 형식 생성
    print(f"Qwen 토크나이저 로드 실패: {e}")
    print("\n수동 ChatML 형식:")
    for msg in messages:
        role = msg['role']
        content = msg['content']
        print(f"<|im_start|>{role}\n{content}<|im_end|>")

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jskim\.cache\huggingface\hub\models--Qwen--Qwen2-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Qwen2 Chat Template 결과:
<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
Python is a high-level programming language.<|im_end|>



In [5]:
# === tokenize=True vs False 차이 ===
# tokenize=False: 문자열 반환 (형식 확인용)
# tokenize=True: 토큰 ID 리스트 반환 (모델 입력용)

try:
    # 문자열로 확인
    text_result = qwen_tok.apply_chat_template(messages, tokenize=False)
    print("tokenize=False (문자열):")
    print(f"  길이: {len(text_result)} 문자")
    print(f"  내용: {text_result[:100]}...")
    
    # 토큰 ID로 변환
    token_result = qwen_tok.apply_chat_template(messages, tokenize=True)
    print(f"\ntokenize=True (토큰 ID):")
    print(f"  길이: {len(token_result)} 토큰")
    print(f"  처음 10개: {token_result[:10]}")
except NameError:
    print("Qwen 토크나이저가 없어서 스킵합니다.")
    print("핵심: tokenize=False는 문자열, True는 토큰 ID를 반환")

tokenize=False (문자열):
  길이: 181 문자
  내용: <|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
What is Python?<|im_en...

tokenize=True (토큰 ID):
  길이: 2 토큰
  처음 10개: [Encoding(num_tokens=34, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]


---
## 3. Alpaca → ShareGPT → Chat Template 변환

Phase 2에서 봤던 Alpaca 형식을 파인튜닝에 쓸 수 있는 형태로 단계별 변환한다.

In [6]:
# === 1단계: Alpaca → ShareGPT (messages 리스트) 변환 ===

def alpaca_to_messages(sample):
    """
    Alpaca 형식의 단일 샘플을 messages 리스트로 변환.
    input이 있으면 instruction에 붙여서 user 메시지로 만든다.
    """
    messages = []
    
    # user 메시지 구성
    if sample.get('input') and sample['input'].strip():
        user_content = f"{sample['instruction']}\n\nInput: {sample['input']}"
    else:
        user_content = sample['instruction']
    
    messages.append({"role": "user", "content": user_content})
    messages.append({"role": "assistant", "content": sample['output']})
    
    return {"messages": messages}

# 테스트
alpaca_samples = [
    {
        "instruction": "Summarize the following text.",
        "input": "Machine learning is a subset of AI that enables systems to learn from data.",
        "output": "ML is an AI subset that learns from data.",
    },
    {
        "instruction": "Write a Python function to reverse a string.",
        "input": "",
        "output": "def reverse(s):\n    return s[::-1]",
    },
]

print("Alpaca -> Messages 변환 결과:")
print("=" * 60)
for i, sample in enumerate(alpaca_samples):
    result = alpaca_to_messages(sample)
    has_input = "input 있음" if sample['input'] else "input 없음"
    print(f"\n[{i}] ({has_input})")
    for msg in result['messages']:
        role = msg['role']
        content = msg['content'][:60]
        print(f"  {role}: {content}...")

Alpaca -> Messages 변환 결과:

[0] (input 있음)
  user: Summarize the following text.

Input: Machine learning is a ...
  assistant: ML is an AI subset that learns from data....

[1] (input 없음)
  user: Write a Python function to reverse a string....
  assistant: def reverse(s):
    return s[::-1]...


In [7]:
# === 2단계: Messages → Chat Template 적용 ===
# 변환된 messages를 모델별 형식으로 포맷팅

def messages_to_chatml(messages):
    """messages 리스트를 ChatML 형식 문자열로 변환"""
    result = ""
    for msg in messages:
        role = msg['role']
        content = msg['content']
        result += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    return result.strip()

def messages_to_llama3(messages):
    """messages 리스트를 Llama 3 형식 문자열로 변환"""
    result = "<|begin_of_text|>"
    for msg in messages:
        role = msg['role']
        content = msg['content']
        result += f"<|start_header_id|>{role}<|end_header_id|>\n\n{content}<|eot_id|>"
    return result

# Alpaca 샘플을 두 형식으로 변환
sample = alpaca_samples[0]
msgs = alpaca_to_messages(sample)['messages']

print("ChatML 형식:")
print(messages_to_chatml(msgs))
print("\n" + "=" * 60)
print("\nLlama 3 형식:")
print(messages_to_llama3(msgs))

ChatML 형식:
<|im_start|>user
Summarize the following text.

Input: Machine learning is a subset of AI that enables systems to learn from data.<|im_end|>
<|im_start|>assistant
ML is an AI subset that learns from data.<|im_end|>


Llama 3 형식:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Summarize the following text.

Input: Machine learning is a subset of AI that enables systems to learn from data.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

ML is an AI subset that learns from data.<|eot_id|>


In [8]:
# === 실전: Alpaca 데이터셋 전체를 messages 형식으로 변환 ===
dataset = load_dataset("tatsu-lab/alpaca", trust_remote_code=True)
train = dataset['train']

# 샘플 100개만 변환 (전체는 시간이 오래 걸림)
small = train.select(range(100))
converted = small.map(alpaca_to_messages)

print(f"변환 완료: {len(converted)}개")
print(f"새로운 열: {converted.column_names}")
print(f"\n샘플 확인:")
for msg in converted[0]['messages']:
    role = msg['role']
    content = msg['content'][:80]
    print(f"  {role}: {content}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'tatsu-lab/alpaca' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Map: 100%|██████████| 100/100 [00:00<00:00, 3677.92 examples/s]

변환 완료: 100개
새로운 열: ['instruction', 'input', 'output', 'text', 'messages']

샘플 확인:
  user: Give three tips for staying healthy.
  assistant: 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 


---
## 4. 멀티턴 대화 데이터 구성

싱글턴(Alpaca)은 1 질문 → 1 응답이지만, 챗봇은 여러 턴의 대화가 필요하다.

In [9]:
# === 멀티턴 대화 데이터 예시 ===
# ShareGPT 형식: conversations 리스트에 여러 턴이 들어감

multi_turn_examples = [
    {
        "conversations": [
            {"from": "human", "value": "What is a neural network?"},
            {"from": "gpt", "value": "A neural network is a computing system inspired by biological neural networks."},
            {"from": "human", "value": "Can you explain backpropagation?"},
            {"from": "gpt", "value": "Backpropagation is the algorithm used to calculate gradients for updating weights."},
            {"from": "human", "value": "What's the vanishing gradient problem?"},
            {"from": "gpt", "value": "When gradients become very small during backprop, early layers stop learning effectively."},
        ]
    },
]

def sharegpt_to_messages(sample):
    """
    ShareGPT 형식을 표준 messages 형식으로 변환.
    from: human -> role: user
    from: gpt   -> role: assistant
    """
    role_map = {"human": "user", "gpt": "assistant"}
    messages = []
    for turn in sample['conversations']:
        role = role_map.get(turn['from'], turn['from'])
        messages.append({"role": role, "content": turn['value']})
    return {"messages": messages}

# 변환
example = multi_turn_examples[0]
result = sharegpt_to_messages(example)

turn_count = len(result['messages'])
print(f"멀티턴 대화 ({turn_count}턴):")
print("=" * 60)
for msg in result['messages']:
    role = msg['role']
    content = msg['content'][:70]
    print(f"  [{role}] {content}")

print(f"\n-> 이 대화를 Chat Template 적용하면 모델 학습 입력이 됨")
print(f"\nChatML 적용:")
print(messages_to_chatml(result['messages']))

멀티턴 대화 (6턴):
  [user] What is a neural network?
  [assistant] A neural network is a computing system inspired by biological neural n
  [user] Can you explain backpropagation?
  [assistant] Backpropagation is the algorithm used to calculate gradients for updat
  [user] What's the vanishing gradient problem?
  [assistant] When gradients become very small during backprop, early layers stop le

-> 이 대화를 Chat Template 적용하면 모델 학습 입력이 됨

ChatML 적용:
<|im_start|>user
What is a neural network?<|im_end|>
<|im_start|>assistant
A neural network is a computing system inspired by biological neural networks.<|im_end|>
<|im_start|>user
Can you explain backpropagation?<|im_end|>
<|im_start|>assistant
Backpropagation is the algorithm used to calculate gradients for updating weights.<|im_end|>
<|im_start|>user
What's the vanishing gradient problem?<|im_end|>
<|im_start|>assistant
When gradients become very small during backprop, early layers stop learning effectively.<|im_end|>


---
## 5. 토크나이저별 특수 토큰 비교

In [10]:
# === 모델별 특수 토큰 비교 ===
# 모델마다 BOS, EOS, PAD 토큰이 다르다

model_names = [
    ('gpt2', 'GPT-2'),
]

# 추가 모델이 있으면 로드 시도
optional_models = [
    ('Qwen/Qwen2-0.5B-Instruct', 'Qwen2'),
]

print("모델별 특수 토큰:")
print("=" * 70)

for model_id, display_name in model_names + optional_models:
    try:
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        bos = str(tok.bos_token) if tok.bos_token else "(없음)"
        eos = str(tok.eos_token) if tok.eos_token else "(없음)"
        pad = str(tok.pad_token) if tok.pad_token else "(없음)"
        vocab = tok.vocab_size
        has_template = hasattr(tok, 'chat_template') and tok.chat_template is not None
        template_str = "있음" if has_template else "없음"
        print(f"\n  {display_name}:")
        print(f"    BOS: {bos}")
        print(f"    EOS: {eos}")
        print(f"    PAD: {pad}")
        print(f"    vocab_size: {vocab:,}")
        print(f"    chat_template: {template_str}")
    except Exception as e:
        print(f"\n  {display_name}: 로드 실패 ({e})")

print("\n-> GPT-2는 chat template이 없음 (사전학습만 된 모델)")
print("   Qwen2-Instruct는 있음 (instruction 파인튜닝된 모델)")
print("   파인튜닝 시 반드시 대상 모델의 template을 사용해야 함")

모델별 특수 토큰:

  GPT-2:
    BOS: <|endoftext|>
    EOS: <|endoftext|>
    PAD: (없음)
    vocab_size: 50,257
    chat_template: 없음

  Qwen2:
    BOS: (없음)
    EOS: <|im_end|>
    PAD: <|endoftext|>
    vocab_size: 151,643
    chat_template: 있음

-> GPT-2는 chat template이 없음 (사전학습만 된 모델)
   Qwen2-Instruct는 있음 (instruction 파인튜닝된 모델)
   파인튜닝 시 반드시 대상 모델의 template을 사용해야 함


---
## 6. 변환 파이프라인 통합

In [11]:
# === Alpaca 데이터 -> ChatML 형식 완전 변환 파이프라인 ===

def full_conversion_pipeline(dataset, format_type='chatml'):
    """
    Alpaca 데이터셋을 지정한 형식의 텍스트로 변환하는 전체 파이프라인.
    
    1. Alpaca -> messages 리스트
    2. messages -> chat template 문자열
    """
    format_funcs = {
        'chatml': messages_to_chatml,
        'llama3': messages_to_llama3,
    }
    
    format_func = format_funcs[format_type]
    
    def convert(sample):
        # 1단계: Alpaca -> messages
        msgs = alpaca_to_messages(sample)['messages']
        # 2단계: messages -> formatted text
        text = format_func(msgs)
        return {"text": text}
    
    return dataset.map(convert)

# 실행
chatml_dataset = full_conversion_pipeline(small, 'chatml')
llama3_dataset = full_conversion_pipeline(small, 'llama3')

print("변환 완료!")
print(f"  ChatML: {len(chatml_dataset)}개")
print(f"  Llama3: {len(llama3_dataset)}개")

print("\nChatML 샘플:")
print(chatml_dataset[0]['text'][:200])
print("\n...")
print("\n" + "=" * 60)
print("\nLlama3 샘플:")
print(llama3_dataset[0]['text'][:200])
print("\n...")

Map: 100%|██████████| 100/100 [00:00<00:00, 4158.79 examples/s]

변환 완료!
  ChatML: 100개
  Llama3: 100개

ChatML 샘플:
<|im_start|>user
Give three tips for staying healthy.<|im_end|>
<|im_start|>assistant
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep you

...


Llama3 샘플:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Give three tips for staying healthy.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

1.Eat a balanced diet and make sure to include p

...


---
## 정리

| 개념 | 핵심 |
|------|------|
| **Chat Template** | 모델이 역할을 구분하는 약속된 형식 |
| **apply_chat_template()** | 토크나이저 내장 템플릿으로 자동 변환 |
| **Alpaca → Messages** | input 유무에 따라 user 메시지 구성 |
| **ShareGPT → Messages** | human→user, gpt→assistant 매핑 |
| **멀티턴** | 챗봇 파인튜닝에 필수, 여러 턴의 대화 |
| **핵심 원칙** | 파인튜닝 대상 모델의 template에 맞춰야 함 |